In [ ]:
from __future__ import annotations
import argparse
import os
import pandas as pd
import numpy as np
from pandas.api.types import CategoricalDtype

In [ ]:
# --- load ---------------------------------------------------
# raw_data = pd.read_csv(r"C:\Users\miked\Desktop2\RIT ISTE Data Mining\project\investigator_nacc70.csv")
# m = pd.read_excel("variable_mapping.xlsx")

In [ ]:
# # drop composite == "remove" FIRST to create proc_data
# cols_to_remove = m.loc[m["composite"] == "remove", "Variable"].dropna().tolist()
# proc_data = raw_data.drop(columns=[c for c in cols_to_remove if c in raw_data.columns], errors="ignore")
# print(f"removed {len([c for c in cols_to_remove if c in raw_data.columns])} composite columns")

# Cleaning Functions

In [ ]:
EXPLICIT_A5_BACKFILL_VARS = [
    "TOBAC30","TOBAC100","SMOKYRS","PACKSPER","QUITSMOK",
    "ALCOCCAS","ALCFREQ",
    "CVHATT","HATTMULT","HATTYEAR","CVOTHR",
    "CBSTROKE","STROKMUL","NACCSTYR",
    "CBTIA","TIAMULT","NACCTIYR",
    "PD","PDYR","PDOTHR","PDOTHRYR",
    "SEIZURES",
    "NACCTBI","TBI","TBIBRIEF","TBIEXTEN","TBIWOLOS","TBIYEAR",
    "ALCOHOL","ABUSOTHR","ABUSX",
    "PTSD","BIPOLAR","SCHIZ","DEP2YRS","DEPOTHR","ANXIETY","OCD",
    "NPSYDEV","PSYCDIS","PSYCDISX"
]


whitelist_x = {
    "sex","insex","naccaanx","arthupex","arthloex","anx",
    "naccnrex","beanx","whodiddx","bipoldx","ptsddx",
    "artupex","artloex","cancsite"
}

npiq_sev_cols = [
        "DELSEV", "HALLSEV", "AGITSEV", "DEPDSEV",
        "ANXSEV", "ELATSEV", "APASEV", "DISNSEV",
        "IRRSEV", "MOTSEV", "NITESEV", "APPSEV",
    ]

npiq_screener_map = {
        "DELSEV":  "DEL",
        "HALLSEV": "HALL",
        "AGITSEV": "AGIT",
        "DEPDSEV": "DEPD",
        "ANXSEV":  "ANX",
        "ELATSEV": "ELAT",
        "APASEV":  "APA",
        "DISNSEV": "DISN",
        "IRRSEV":  "IRR",
        "MOTSEV":  "MOT",
        "NITESEV": "NITE",
        "APPSEV":  "APP",
    }

In [ ]:
#is it statisticall risky to note all 0 when the score_nonres is out of range of the score field? so when it is 998 or something?

In [ ]:
def clean_placeholders(df: pd.DataFrame) -> pd.DataFrame:
    """Replace placeholders like '.', '-4', or '-4.4' with NaN."""
    placeholders = ["", "na", "n/a", "nan", "null", ".", "-4", "-4.4"]
    df = df.replace([-4, "-4", -4.4, "-4.4"], np.nan)
    for c in df.columns:
        if df[c].dtype == object or pd.api.types.is_string_dtype(df[c]):
            df[c] = df[c].astype(str).str.strip()
            df.loc[df[c].str.lower().isin(placeholders), c] = np.nan
    return df


def fix_dtypes(df: pd.DataFrame, m: pd.DataFrame) -> pd.DataFrame:
    """
    Convert variables to numeric, string, or category types using mapping file.
    Ignores dtype_conve == 'score_nonres' because those are already forced numeric earlier.
    """
    m["dtype_conve"] = m["dtype_conve"].astype(str).str.lower().str.strip()

    # exclude variables that are known numeric but may have categorial missing assignment (i.e. 95=physical limitation)
    score_cols = set(m.loc[m["dtype_conve"] == "score_nonres", "Variable"])

    num_cols = [c for c in m.loc[m["dtype_conve"] == "num", "Variable"] if c in df.columns and c not in score_cols]
    char_cols = [c for c in m.loc[m["dtype_conve"] == "char", "Variable"] if c in df.columns and c not in score_cols]
    fact_cols = [c for c in m.loc[m["dtype_conve"] == "factor", "Variable"] if c in df.columns and c not in score_cols]

    for c in num_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    for c in char_cols:
        df[c] = df[c].astype("string")
    for c in fact_cols:
        df[c] = df[c].astype("string").astype("category")

    return df


def drop_text_x_columns(df: pd.DataFrame,
                        whitelist: set[str] | None = None,
                        suffixes: tuple[str, ...] = ("x",)) -> pd.DataFrame:
    """
    drop columns whose names end with given suffixes (default: 'x'),
    except those explicitly whitelisted (case-insensitive). those columns that end in 'x' are "other" free text fields
    """
    if whitelist is None:
        whitelist = set()

    wl_norm = {w.strip().lower() for w in whitelist}
    suff_norm = tuple(s.lower() for s in suffixes)

    to_drop = []
    for c in df.columns:
        c_norm = str(c).strip().lower()
        if c_norm.endswith(suff_norm) and c_norm not in wl_norm:
            to_drop.append(c)

    if to_drop:
        df = df.drop(columns=to_drop, errors="ignore")

    return df


def flag_admin_and_missing(df: pd.DataFrame, m: pd.DataFrame,
                           invalid_prefix: str = "invalid_") -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    For ALL columns EXCEPT those with dtype_conve == 'score_nonres':
      - If value is one of the admin invalid amounts, set to null (except HRATE)
      - Create Int8 flag invalid_<col>: 1 if an admin invalid value was found, 0 if not, <NA> if missing
      - HRATE is fully excluded from nulling/flagging of these admin codes because it can be 88

    Returns: (df, admin_report)
    """
    admin_invalid_nums = {
        99, 999,
        88, 88.8, 888, 888.8, 8888,
        95, 96, 97, 98,
        995, 996, 997, 998
    }

    # columns to exclude (handled by apply_score_nonres_bounds)
    score_cols = set(
        m.loc[m["dtype_conve"].astype(str).str.strip().str.lower() == "score_nonres", "Variable"]
         .astype(str)
         .tolist()
    )

    df = df.copy()
    flags = {}
    rows = []

    for c in df.columns:
        if c in score_cols:
            continue  # skip score_nonres completely here

        if c == "HRATE":
            continue  # do not touch HRATE for admin invalids

        s = df[c]
        if not (pd.api.types.is_numeric_dtype(s) or pd.api.types.is_string_dtype(s)):
            continue

        vals = pd.to_numeric(s, errors="coerce")
        obs = vals.notna()
        admin_mask = vals.isin(admin_invalid_nums)

        # null out ONLY admin invalids
        if admin_mask.any():
            df.loc[admin_mask, c] = np.nan

        # build flag: 1 if admin invalid, 0 if observed & not admin, <NA> if missing
        f = pd.Series(pd.NA, index=df.index, dtype="Int8")
        f.loc[obs & ~admin_mask] = 0
        f.loc[admin_mask] = 1
        flags[f"{invalid_prefix}{c}"] = f

        if admin_mask.any():
            rows.append({
                "variable": c,
                "admin_invalid_frac": float(admin_mask.mean())
            })

    if flags:
        df = pd.concat([df, pd.DataFrame(flags, index=df.index)], axis=1)

    admin_report = pd.DataFrame(rows).sort_values("admin_invalid_frac", ascending=False)
    return df, admin_report


def keep_v3_plus(df: pd.DataFrame) -> pd.DataFrame:
    """Keep only UDS forms version 3 or higher."""
    if "FORMVER" in df.columns:
        df = df.loc[df["FORMVER"] >= 3].copy()
    return df


def drop_not_in_v3(df: pd.DataFrame, m: pd.DataFrame) -> pd.DataFrame:
    """Drop variables not included in UDS v3 based on mapping file."""
    if "v3" in m.columns and "Variable" in m.columns:
        to_drop = [c for c in m.loc[m["v3"] == 0, "Variable"] if c in df.columns]
        df = df.drop(columns=to_drop, errors="ignore")
    return df


def combine_img_fields(df: pd.DataFrame) -> pd.DataFrame:
    """Combine NACCNMRI and NACCNAPA into IMG_TOT and drop originals."""
    if "NACCNMRI" not in df.columns or "NACCNAPA" not in df.columns:
        return df
    df = df.copy()
    df["IMG_TOT"] = df[["NACCNMRI", "NACCNAPA"]].sum(axis=1, min_count=1)
    df = df.drop(columns=["NACCNMRI", "NACCNAPA"], errors="ignore")
    return df


def backfill_a5_from_mapping_df(
    df: pd.DataFrame,
    mapping_df: pd.DataFrame | None = None,  
    *,
    id_col: str = "NACCID",
    packet_col: str = "PACKET",
    donor_packets: tuple[str, ...] = ("IT", "I"),      # baseline sources, in priority order
    target_packets: tuple[str, ...] = ("F", "FT", "T"),# follow-ups to fill
    explicit_vars: list[str] | None = EXPLICIT_A5_BACKFILL_VARS,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Carry forward baseline values for the given explicit vars from subject packet(s)
    to missing values in target packet(s), per participant id.

    - Only columns present in df are processed.
    - Respects category dtype round-trip.
    - Safe if some packets or vars are absent.
    """
    if explicit_vars is None or len(explicit_vars) == 0:
        if verbose:
            print("backfill_a5_from_mapping_df: no explicit_vars provided; nothing to do.")
        return df

    cols = [c for c in explicit_vars if c in df.columns]
    if not cols:
        if verbose:
            print("backfill_a5_from_mapping_df: none of the explicit vars are in df; skipping.")
        return df

    df = df.copy()

    # normalize packet labels once
    pkt = df[packet_col].astype(str).str.upper()
    donor_set = {p.upper() for p in donor_packets}
    target_set = {p.upper() for p in target_packets}

    # order donors by priority (e.g., IT before I)
    pref = {p: i for i, p in enumerate(donor_packets)}
    df["_pkt_order"] = pkt.map(pref)

    donor_mask = pkt.isin(donor_set)
    if not donor_mask.any():
        if verbose:
            print("backfill_a5_from_mapping_df: no donor packets present; skipping.")
        df.drop(columns=["_pkt_order"], errors="ignore", inplace=True)
        return df

    # donor table, sorted so 'first' respects donor priority
    base = df.loc[donor_mask, [id_col, "_pkt_order"] + cols].sort_values([id_col, "_pkt_order"])

    filled_counts = {}

    for v in cols:
        # get first non-null donor value per id for this variable
        donors = base.loc[base[v].notna(), [id_col, v]].groupby(id_col, as_index=True)[v].first()

        was_cat = isinstance(df[v].dtype, CategoricalDtype)
        need = df[v].isna() & pkt.isin(target_set) & df[id_col].isin(donors.index)

        if need.any():
            df.loc[need, v] = df.loc[need, id_col].map(donors)
            if was_cat:
                df[v] = df[v].astype("category")
                try:
                    df[v] = df[v].cat.remove_unused_categories()
                except Exception:
                    pass

        filled_counts[v] = int(need.sum())

    df.drop(columns=["_pkt_order"], errors="ignore", inplace=True)

    if verbose:
        total = sum(filled_counts.values())
        nonzero = {k: v for k, v in filled_counts.items() if v > 0}
        print(f"backfill_a5_from_mapping_df: filled {total} values across {len(cols)} columns.")
        if nonzero:
            top = sorted(nonzero.items(), key=lambda kv: kv[1], reverse=True)[:10]
            print(" top fills:", ", ".join(f"{k}:{v}" for k, v in top))

    return df


def fill_moca(df: pd.DataFrame, moca_col="MOCATOTS", moca_blind_col="MOCBTOTS") -> pd.DataFrame:
    """
    ensures the values of moca's that used the telephone moca are adjusted with the empirically proven conversion equation
    """
    if moca_col not in df.columns or moca_blind_col not in df.columns:
        return df
    mb = pd.to_numeric(df[moca_blind_col], errors="coerce")
    scaled = (mb * 30.0) / 22.0
    scaled = scaled.clip(lower=0, upper=30)
    need = df[moca_col].isna() & mb.notna()
    df.loc[need, moca_col] = scaled[need]
    return df

def clean_npiq_severity(
    df: pd.DataFrame,
    *,
    severity_cols: list[str],
    screener_map: dict[str, str] | None = None
) -> pd.DataFrame:
    """
    Clean NPI-Q severity variables:

      - Coerce to numeric
      - Replace admin codes (8, 9, -4) with NaN
      - If screener_map is given: force severity = NaN when screener != 1
        (NPI-Q severity is only meaningful when parent symptom is present)
    """
    out = df.copy()

    # 1) strip admin codes in severity
    for sev in severity_cols:
        if sev not in out.columns:
            continue
        out[sev] = pd.to_numeric(out[sev], errors="coerce")
        out.loc[out[sev].isin([8, 9, -4]), sev] = np.nan

    # 2) enforce screener logic if mapping is provided
    if screener_map:
        for sev, scr in screener_map.items():
            if sev in out.columns and scr in out.columns:
                # any value other than 1 on screener -> severity is not applicable
                mask_no_symptom = out[scr] != 1
                out.loc[mask_no_symptom, sev] = np.nan

    return out


def add_visit_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add visit_date, days_since_prev, and binary naccudsd_bin variables."""
    if {"VISITYR", "VISITMO", "VISITDAY"}.issubset(df.columns):
        df["visit_date"] = pd.to_datetime(
            {"year": df["VISITYR"], "month": df["VISITMO"], "day": df["VISITDAY"]},
            errors="coerce"
        )
    elif {"VISITYR", "VISITMO"}.issubset(df.columns):
        df["visit_date"] = pd.to_datetime(
            {"year": df["VISITYR"], "month": df["VISITMO"], "day": 1},
            errors="coerce"
        )
    else:
        df["visit_date"] = pd.NaT

    df = df.sort_values(["NACCID", "visit_date"]).reset_index(drop=True)
    df["days_since_prev"] = df.groupby("NACCID")["visit_date"].diff().dt.days
    first = df.groupby("NACCID").cumcount().eq(0)
    df.loc[first, "days_since_prev"] = 0

    y = pd.to_numeric(df["NACCUDSD"], errors="coerce")
    df["naccudsd_bin"] = y.ne(1).where(y.notna()).astype("Int64")
    return df


def drop_visit_parts_after_visit_date(df: pd.DataFrame,
                                      drop_day: bool = True,
                                      drop_month: bool = True,
                                      drop_year: bool = False) -> pd.DataFrame:
    """After constructing visit_date, optionally drop raw date parts to reduce leakage."""
    cols = []
    if drop_day and "VISITDAY" in df.columns:
        cols.append("VISITDAY")
    if drop_month and "VISITMO" in df.columns:
        cols.append("VISITMO")
    if drop_year and "VISITYR" in df.columns:
        cols.append("VISITYR")

    if cols:
        df = df.drop(columns=cols, errors="ignore")
    return df


def apply_score_nonres_bounds(df: pd.DataFrame, m: pd.DataFrame,
                              invalid_prefix: str = "invalid_") -> pd.DataFrame:
    """
    For mapping rows with dtype_conve == 'score_nonres' and numeric [min, max]:
      - Out-of-bounds -> set original to 0
      - In-range      -> keep value
      - Missing       -> keep NaN
      - Create Int8 flag invalid_<col>: 1 if OOB, 0 if in-range, <NA> if missing
      - Force these columns to numeric (float) at the end
    """
    mm = (
        m.loc[m["dtype_conve"].astype(str).str.strip().str.lower() == "score_nonres",
              ["Variable", "min", "max"]]
         .assign(min=lambda x: pd.to_numeric(x["min"], errors="coerce"),
                 max=lambda x: pd.to_numeric(x["max"], errors="coerce"))
         .dropna(subset=["min", "max"])
    )

    bounds = {
        row["Variable"]: (float(row["min"]), float(row["max"]))
        for _, row in mm.iterrows()
        if row["Variable"] in df.columns
    }
    if not bounds:
        return df

    df = df.copy()
    flag_cols = {}

    for col, (lo, hi) in bounds.items():
        vals = pd.to_numeric(df[col], errors="coerce")
        obs = vals.notna()
        inb = obs & (vals >= lo) & (vals <= hi)
        oob = obs & ~inb

        # set OOB to 0
        if oob.any():
            df.loc[oob, col] = 0

        # build flag: 1 if OOB, 0 if in-range, <NA> if missing
        f = pd.Series(pd.NA, index=df.index, dtype="Int8")
        f.loc[inb] = 0
        f.loc[oob] = 1
        flag_cols[f"{invalid_prefix}{col}"] = f

        # ensure numeric after edits
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("float64")

    # add all flags at once
    df = pd.concat([df, pd.DataFrame(flag_cols, index=df.index)], axis=1)
    return df





# Running cleaning functions
Chose to run seperately and not in a main to debug if necessary

In [ ]:
# print("Initial shape:", proc_data.shape)

In [ ]:
# Step 1. Keep only v3+ visits
# proc_data = keep_v3_plus(proc_data)
# print("After keep_v3_plus:", proc_data.shape)

In [ ]:
# # Step 2. Drop variables not in version 3+
# proc_data = drop_not_in_v3(proc_data, m)
# print("After drop_not_in_v3:", proc_data.shape)

In [ ]:
# Step 3. Drop text columns ending in 'X' except whitelisted ones
# proc_data = drop_text_x_columns(proc_data, whitelist=whitelist_x, suffixes=("x",))
# print("After drop_text_x_columns:", proc_data.shape)

In [ ]:
# Step 4. Clean placeholders
# proc_data = clean_placeholders(proc_data)
# print("After clean_placeholders:", proc_data.shape)

In [ ]:
# # Step 5. Apply bounds/meta flags for score_nonres variables
# proc_data = apply_score_nonres_bounds(proc_data, m)
# print("After apply_score_nonres_bounds:", proc_data.shape)

In [ ]:
# Step 6. Flag and null admin invalids (and add invalid_<col> flags)
# proc_data, admin_report = flag_admin_and_missing(proc_data, m)
# print("After flag_admin_and_missing:", proc_data.shape)
# print(admin_report.head())

In [ ]:
# # Step 7. Fix dtypes using mapping file
# proc_data = fix_dtypes(proc_data, m)
# print("After fix_dtypes:", proc_data.shape)

In [ ]:
# # Step 8. Fill Moca totals from blind version
# proc_data = fill_moca(proc_data)
# print("After fill_moca:", proc_data.shape)

In [ ]:
# # Step 9. Backfill A5 variables
# proc_data = backfill_a5_from_mapping_df(proc_data, m)
# print("After backfill_a5_from_mapping_df:", proc_data.shape)

In [ ]:
# # Step 10. Combine imaging fields NACCNMRI + NACCNAPA into IMG_TOT
# proc_data = combine_img_fields(proc_data)
# print("After combine_img_fields:", proc_data.shape)

In [ ]:
# # Step 11. Add visit features (visit_date, days_since_prev, naccudsd_bin)
# proc_data = add_visit_features(proc_data)
# print("After add_visit_features:", proc_data.shape)

In [ ]:
# # Step 12. Drop date parts (VISITMO, VISITDAY) to prevent leakage
# proc_data = drop_visit_parts_after_visit_date(proc_data, drop_day=True, drop_month=True, drop_year=False)
# print("After drop_visit_parts_after_visit_date:", proc_data.shape)

In [ ]:
# # Step 13. Drop sparse columns (>30% missing)
# proc_data = drop_sparse(proc_data, max_missing=0.30)
# print("After drop_sparse:", proc_data.shape)

In [ ]:
# # Step 14. Drop diagnosis-related leaky variables
# proc_data = drop_leaky(proc_data)
# print("After drop_leaky:", proc_data.shape)

In [ ]:
# proc_data = drop_zero_invalid_flags(proc_data)

In [ ]:
# print("\nFinal DataFrame Info:")
# proc_data.info()

# print("\nFinal Shape:", proc_data.shape)
# print("Columns:", len(proc_data.columns))

In [ ]:
# define your output path
#out_path =

# save without the pandas index and handle large fields safely
# proc_data.to_csv(out_path, index=False, encoding="utf-8", quoting=1)


In [ ]:
# ------------------------------------------------------------
# main pipeline in the exact order you requested
# ------------------------------------------------------------
def run_pipeline(drop_visit_year: bool = False):
    """
    loads raw + mapping, drops composite=='remove', then runs:
      1) keep_v3_plus
      2) drop_not_in_v3
      3) drop_text_x_columns (with whitelist_x)
      4) clean_placeholders
      5) apply_score_nonres_bounds  (uses mapping)
      6) flag_admin_and_missing     (excludes score_nonres; special-cases HRATE)
      7) fix_dtypes                 (respects mapping; ignores score_nonres)
      8) fill_moca
      9) clean_npiq_severity        (NPI-Q severity admin codes + screener logic)
     10) backfill_a5_from_mapping_df (explicit A5 list)
     11) combine_img_fields
     12) add_visit_features
     13) drop_visit_parts_after_visit_date
     14) drop_sparse -- Moved to model process
     15) drop_leaky -- Moved to model process
     16) drop_zero_invalid_flags -- Moved to model process

    returns: (raw_data, proc_data, admin_report)
    """
    # --- load ---
    raw_data = pd.read_csv(r"C:\Users\miked\Desktop2\RIT ISTE Data Mining\project\investigator_nacc70.csv")
    m = pd.read_excel("variable_mapping.xlsx")

    # --- composite drop to create proc_data ---
    cols_to_remove = m.loc[m["composite"] == "remove", "Variable"].dropna().tolist()
    present = [c for c in cols_to_remove if c in raw_data.columns]
    proc_data = raw_data.drop(columns=present, errors="ignore")
    print(f"removed {len(present)} composite columns")

    print("initial shape:", proc_data.shape)

    # 1) keep only v3+
    proc_data = keep_v3_plus(proc_data)
    print("after keep_v3_plus:", proc_data.shape)

    # 2) drop vars not in v3
    proc_data = drop_not_in_v3(proc_data, m)
    print("after drop_not_in_v3:", proc_data.shape)

    # 3) drop free-text *x (with whitelist)
    proc_data = drop_text_x_columns(proc_data, whitelist=whitelist_x, suffixes=("x",))
    print("after drop_text_x_columns:", proc_data.shape)

    # 4) clean placeholders to NaN (true missing only)
    proc_data = clean_placeholders(proc_data)
    print("after clean_placeholders:", proc_data.shape)

    # 5) fill moca (blind -> scaled total)
    proc_data = fill_moca(proc_data)
    print("after fill_moca:", proc_data.shape)

    # 6) clean NPI-Q severity (admin codes + screener logic)
    proc_data = clean_npiq_severity(
        proc_data,
        severity_cols=npiq_sev_cols,
        screener_map=npiq_screener_map,
    )
    print("after clean_npiq_severity:", proc_data.shape)

    # 7) backfill a5 from baseline to follow-ups (explicit list)
    proc_data = backfill_a5_from_mapping_df(
        proc_data, m,
        explicit_vars=EXPLICIT_A5_BACKFILL_VARS
    )
    print("after backfill_a5_from_mapping_df:", proc_data.shape)

    # 8) score_nonres bounds -> invalid_<col> flags + set OOB to 0 (keep numeric)
    proc_data = apply_score_nonres_bounds(proc_data, m)
    print("after apply_score_nonres_bounds:", proc_data.shape)
    
    # 9) admin invalids (exclude score_nonres vars; HRATE exempt)
    proc_data, admin_report = flag_admin_and_missing(proc_data, m)
    print("after flag_admin_and_missing:", proc_data.shape)
   
    # 10) fix dtypes (num/char/factor only; leaves score_nonres alone)
    proc_data = fix_dtypes(proc_data, m)
    print("after fix_dtypes:", proc_data.shape)
    
    # 11) combine imaging fields
    proc_data = combine_img_fields(proc_data)
    print("after combine_img_fields:", proc_data.shape)

    # 12) visit features
    proc_data = add_visit_features(proc_data)
    print("after add_visit_features:", proc_data.shape)

    # 13) drop raw visit parts (day/month yes; year optional)
    proc_data = drop_visit_parts_after_visit_date(
        proc_data,
        drop_day=True,
        drop_month=True,
        drop_year=bool(drop_visit_year)
    )
    print("after drop_visit_parts_after_visit_date:", proc_data.shape)

    # # 14) drop sparse (>30% missing)
    # proc_data = drop_sparse(proc_data, max_missing=0.30)
    # print("after drop_sparse:", proc_data.shape)

    # # 15) drop leaky target columns
    # proc_data = drop_leaky(proc_data)
    # print("after drop_leaky:", proc_data.shape)

    # # 16) drop invalid_* flags that are all zeros
    # proc_data = drop_zero_invalid_flags(proc_data, prefix="invalid_")
    # print("after drop_zero_invalid_flags:", proc_data.shape)

    # keep the admin_report attached for convenience
    proc_data.attrs["admin_report"] = admin_report

    print(f"Final Shape: {proc_data.shape}")
    return raw_data, proc_data, admin_report
